# ERTime - Linear Regression Model (Baseline)
## Emergency Room Wait Time Prediction
**AAI 595 - Applied Machine Learning**

Michael Lugo, Amoy Mowatt, Ardit Cana, Wesley Nabo

This notebook trains a **Linear Regression** model to predict ER wait times using hospital operational data.

In [ ]:
# Essential imports for ER wait time prediction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
print("✅ All imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Model: Linear Regression (Baseline)")

## Load and Explore Dataset

In [ ]:
# Load the cleaned ER dataset
df = pd.read_csv("cleaned_data.csv")

print(f"✅ Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  • {col} ({df[col].dtype})")

print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Dataset statistics
print("Dataset Statistics:")
df.describe()

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(15, 10))

# Plot 1: Distribution of Target Variable
plt.subplot(2, 3, 1)
plt.hist(df["Total Wait Time (min)"], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
plt.xlabel("Total Wait Time (min)")
plt.ylabel("Frequency")
plt.title("Distribution of ER Wait Times")
plt.grid(True, alpha=0.3)

# Plot 2: Wait Time by Urgency Level
plt.subplot(2, 3, 2)
sns.boxplot(data=df, x="Urgency Level", y="Total Wait Time (min)",
            order=["Low", "Medium", "High", "Critical"])
plt.title("Wait Time by Urgency Level")
plt.grid(True, alpha=0.3)

# Plot 3: Wait Time by Region
plt.subplot(2, 3, 3)
sns.boxplot(data=df, x="Region", y="Total Wait Time (min)")
plt.title("Wait Time by Region")
plt.grid(True, alpha=0.3)

# Plot 4: Wait Time by Time of Day
plt.subplot(2, 3, 4)
sns.boxplot(data=df, x="Time of Day", y="Total Wait Time (min)")
plt.xticks(rotation=45)
plt.title("Wait Time by Time of Day")
plt.grid(True, alpha=0.3)

# Plot 5: Wait Time by Season
plt.subplot(2, 3, 5)
sns.boxplot(data=df, x="Season", y="Total Wait Time (min)")
plt.title("Wait Time by Season")
plt.grid(True, alpha=0.3)

# Plot 6: Correlation Heatmap (numeric features only)
plt.subplot(2, 3, 6)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Feature Correlation Heatmap")

plt.tight_layout()
plt.show()

print("✅ Exploratory Data Analysis complete!")

## Feature Selection and Preprocessing

**Important:** We only use features available **before/at arrival**.

`Time to Registration`, `Time to Triage`, and `Time to Medical Professional` are process durations that literally sum to `Total Wait Time` — including them would be **data leakage** (R² = 1.0, meaningless).

In [ ]:
# Select features available BEFORE/AT arrival
feature_cols = [
    "Region", "Day of Week", "Season", "Time of Day", "Urgency Level",
    "Nurse-to-Patient Ratio", "Specialist Availability", "Facility Size (Beds)"
]

X = df[feature_cols].copy()
y = df["Total Wait Time (min)"].copy()

print(f"Feature columns selected: {len(feature_cols)}")
for col in feature_cols:
    print(f"  • {col}")
print(f"\nTarget variable: Total Wait Time (min)")
print(f"  Mean: {y.mean():.2f} minutes")
print(f"  Std:  {y.std():.2f} minutes")
print(f"  Min:  {y.min()} minutes")
print(f"  Max:  {y.max()} minutes")

## Encode Categorical Variables

In [ ]:
# Encode categorical features using LabelEncoder
label_encoders = {}
categorical_cols = X.select_dtypes(include=["object", "string"]).columns

print("Encoding categorical variables...")
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"  ✅ '{col}': {mapping}")

print(f"\n✅ Feature matrix shape: {X.shape}")
print(f"✅ Target variable shape: {y.shape}")

## Standardize Features

In [ ]:
# Standardize numerical features for better model performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Features standardized using StandardScaler")
print(f"Feature means (should be ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Feature stds (should be ~1):  {X_scaled.std(axis=0).round(4)}")

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"✅ Data split complete!")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nTraining target - Mean: {y_train.mean():.2f}, Std: {y_train.std():.2f}")
print(f"Testing target  - Mean: {y_test.mean():.2f}, Std: {y_test.std():.2f}")

## Train the Linear Regression Model

In [ ]:
print("Training Linear Regression model...")
print("=" * 50)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("✅ Linear Regression model trained successfully!")
print(f"Intercept: {lr_model.intercept_:.4f}")
print(f"Number of coefficients: {len(lr_model.coef_)}")

## Make Predictions

In [ ]:
y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

print("✅ Predictions generated!")
print(f"\nSample predictions vs actual (first 10 test samples):")
print(f"{'Actual':>10} {'Predicted':>10} {'Error':>10}")
print("-" * 35)
for i in range(10):
    actual = y_test.iloc[i]
    predicted = y_test_pred[i]
    error = actual - predicted
    print(f"{actual:>10.1f} {predicted:>10.1f} {error:>10.1f}")

## Evaluation Metrics

In [ ]:
# Training metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# Testing metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("=" * 55)
print("LINEAR REGRESSION - EVALUATION METRICS")
print("=" * 55)

print(f"\n{'Metric':<30} {'Training':>12} {'Testing':>12}")
print("-" * 55)
print(f"{'Mean Absolute Error (MAE)':<30} {train_mae:>12.2f} {test_mae:>12.2f}")
print(f"{'Mean Squared Error (MSE)':<30} {train_mse:>12.2f} {test_mse:>12.2f}")
print(f"{'Root Mean Squared Error (RMSE)':<30} {train_rmse:>12.2f} {test_rmse:>12.2f}")
print(f"{'R² Score':<30} {train_r2:>12.4f} {test_r2:>12.4f}")

# Diagnose the model
gap = train_r2 - test_r2
print(f"\n📊 Diagnostics:")
print(f"  Train-Test R² Gap: {gap:.4f}")
if gap > 0.1:
    print(f"  ⚠️  Possible overfitting (gap > 0.1)")
elif test_r2 < 0.5:
    print(f"  ⚠️  Possible underfitting (low R² score)")
else:
    print(f"  ✅ Model appears to generalize well")

## Cross-Validation

In [ ]:
print("Performing 5-Fold Cross-Validation...")
print("=" * 50)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr_model, X_scaled, y, cv=kf, scoring='r2')

for fold, score in enumerate(cv_scores, 1):
    print(f"  Fold {fold}: R² = {score:.4f}")

print(f"\n✅ Cross-Validation Results:")
print(f"  Mean R²: {cv_scores.mean():.4f}")
print(f"  Std R²:  {cv_scores.std():.4f}")
print(f"  95% CI:  [{cv_scores.mean() - 1.96*cv_scores.std():.4f}, {cv_scores.mean() + 1.96*cv_scores.std():.4f}]")

## Feature Importance (Coefficients)

In [ ]:
coeff_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lr_model.coef_
}).sort_values("Coefficient", key=abs, ascending=False)

print("Feature Coefficients (sorted by absolute value):")
print("=" * 50)
for _, row in coeff_df.iterrows():
    direction = "↑" if row["Coefficient"] > 0 else "↓"
    print(f"  {direction} {row['Feature']:<30} {row['Coefficient']:>10.4f}")

## Comprehensive Visualizations

In [ ]:
plt.figure(figsize=(18, 12))

# Plot 1: Actual vs Predicted (Training)
plt.subplot(2, 3, 1)
plt.scatter(y_train, y_train_pred, alpha=0.3, edgecolors='k', linewidths=0.3, s=20, color='steelblue')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel("Actual Wait Time (min)")
plt.ylabel("Predicted Wait Time (min)")
plt.title(f"Training: Actual vs Predicted (R²={train_r2:.4f})")
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Actual vs Predicted (Testing)
plt.subplot(2, 3, 2)
plt.scatter(y_test, y_test_pred, alpha=0.4, edgecolors='k', linewidths=0.3, s=20, color='coral')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel("Actual Wait Time (min)")
plt.ylabel("Predicted Wait Time (min)")
plt.title(f"Testing: Actual vs Predicted (R²={test_r2:.4f})")
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Residuals Distribution
plt.subplot(2, 3, 3)
residuals = y_test - y_test_pred
plt.hist(residuals, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Frequency")
plt.title(f"Residuals Distribution (Mean={residuals.mean():.2f})")
plt.grid(True, alpha=0.3)

# Plot 4: Residuals vs Predicted
plt.subplot(2, 3, 4)
plt.scatter(y_test_pred, residuals, alpha=0.4, edgecolors='k', linewidths=0.3, s=20, color='green')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel("Predicted Wait Time (min)")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted Values")
plt.grid(True, alpha=0.3)

# Plot 5: Feature Coefficients
plt.subplot(2, 3, 5)
coeff_sorted = coeff_df.sort_values("Coefficient", key=abs, ascending=True)
colors_bar = ['green' if c > 0 else 'red' for c in coeff_sorted["Coefficient"]]
plt.barh(coeff_sorted["Feature"], coeff_sorted["Coefficient"], color=colors_bar)
plt.xlabel("Coefficient Value")
plt.title("Feature Coefficients")
plt.grid(True, alpha=0.3)

# Plot 6: Cross-Validation Scores
plt.subplot(2, 3, 6)
folds = range(1, len(cv_scores) + 1)
plt.bar(folds, cv_scores, alpha=0.7, color='steelblue', edgecolor='black')
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--',
            linewidth=2, label=f'Mean R²={cv_scores.mean():.4f}')
plt.xlabel("Fold")
plt.ylabel("R² Score")
plt.title("5-Fold Cross-Validation Scores")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ All visualizations complete!")

## Model Summary

In [ ]:
print("=" * 50)
print("LINEAR REGRESSION MODEL - SUMMARY")
print("=" * 50)
print(f"  Dataset:         {df.shape[0]} samples, {len(feature_cols)} features")
print(f"  Train/Test:      {X_train.shape[0]}/{X_test.shape[0]} (80/20 split)")
print(f"  Test MAE:        {test_mae:.2f} minutes")
print(f"  Test RMSE:       {test_rmse:.2f} minutes")
print(f"  Test R²:         {test_r2:.4f}")
print(f"  CV Mean R²:      {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Top Feature:     {coeff_df.iloc[0]['Feature']}")
print("=" * 50)